<a href="https://colab.research.google.com/github/piyushanangude/Codetech/blob/main/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ====================================================
# ETL PIPELINE PROJECT
# Dataset: customer_sales_dataset.csv
# Tools: Pandas + Scikit-Learn
# ====================================================

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ------------------------------
# STEP 1: EXTRACT
# ------------------------------

DATA_PATH = "/content/customer_sales_dataset.csv"
OUTPUT_PATH = "processed_customer_sales.csv"

print("Loading dataset...")
df = pd.read_csv(DATA_PATH)

print("\nDataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

# ------------------------------
# STEP 2: CHECK DATA
# ------------------------------

print("\nMissing Values:")
print(df.isnull().sum())

# ------------------------------
# STEP 3: SPLIT FEATURES & TARGET
# ------------------------------

target_column = "Target"

X = df.drop(columns=[target_column])
y = df[target_column]

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

print("\nNumerical Columns:", list(num_cols))
print("Categorical Columns:", list(cat_cols))

# ------------------------------
# STEP 4: CREATE TRANSFORMATION PIPELINE
# ------------------------------

num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# ------------------------------
# STEP 5: TRANSFORM DATA
# ------------------------------

print("\nApplying preprocessing...")
X_transformed = preprocessor.fit_transform(X)

# Convert to DataFrame
X_processed = pd.DataFrame(
    X_transformed.toarray() if hasattr(X_transformed, "toarray") else X_transformed
)

# Save processed data
X_processed.to_csv(OUTPUT_PATH, index=False)
print("\nProcessed dataset saved as:", OUTPUT_PATH)

# ------------------------------
# STEP 6: TRAIN-TEST SPLIT
# ------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42
)

# ------------------------------
# STEP 7: MODEL TRAINING
# ------------------------------

print("\nTraining Random Forest Model...")
model = RandomForestClassifier(n_estimators=150, random_state=42)
model.fit(X_train, y_train)

# ------------------------------
# STEP 8: EVALUATION
# ------------------------------

y_pred = model.predict(X_test)

print("\nMODEL PERFORMANCE")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# ------------------------------
# COMPLETE
# ------------------------------

print("\nETL + TRANSFORMATION + LOADING + ML COMPLETED SUCCESSFULLY!")


Loading dataset...

Dataset Shape: (200, 8)

First 5 rows:
   CustomerID  Age  Gender  AnnualIncome  SpendingScore       City  \
0           1   56  Female         63484             35       Pune   
1           2   46    Male        102989             87  Bangalore   
2           3   32  Female         56212             81  Bangalore   
3           4   60    Male         63525             90       Pune   
4           5   25    Male         67202              8       Pune   

   PurchaseAmount  Target  
0           40249       0  
1           39644       1  
2           17407       1  
3           19277       1  
4           28193       0  

Missing Values:
CustomerID        0
Age               0
Gender            0
AnnualIncome      0
SpendingScore     0
City              0
PurchaseAmount    0
Target            0
dtype: int64

Numerical Columns: ['CustomerID', 'Age', 'AnnualIncome', 'SpendingScore', 'PurchaseAmount']
Categorical Columns: ['Gender', 'City']

Applying preprocessing...

P